# ResNet34-UNet + H95 — Colab Training (A100 / L4 / V100 / H100)

**Target hardware:** NVIDIA A100 40 GB (Colab High-RAM). Falls back to L4 / V100 with reduced batch and FP16; supports H100. T4 is rejected.

**Approach:** Clones this repo from GitHub into `/content/RESNET_h95`, runs the `src/` pipeline locally, builds a Colab-tuned config, and calls each stage in process. Model architecture is unchanged (`resnet34_unet`, 2-ch input, 512², 1 class).

**Leakage control:** sample selection is driven by the precomputed **clean pair-hash manifests** (`archive_clean_pairhash_manifest_v1`), so train/val/eval are deduplicated and isolated by `pair_sha256`. See `instructions/leakage_decision_logic.md`.

**Canonical sources:**
- **Code:** GitHub `https://github.com/chautuanminh/RESNET_h95` → cloned to `/content/RESNET_h95`
- **Dataset:** DocTamper LMDBs at `/content/drive/MyDrive/usth/Final_Project/dtset/archive`
- **Clean manifests:** `/content/drive/Othercomputers/MSI/Approach/ASCFORMER_full/archive_clean_pairhash_manifest_v1`
- **Encoder cache / checkpoints / results:** Google Drive

**Outputs** stream to `/content/drive/Othercomputers/MSI/results/RESNET_h95/runs/<RUN_DIR>/`.

**One switch — `RUN_MODE`** (section 1): `smoke` | `train` | `resume` | `post_train` | `full`.

## 1. Mount Drive, paths & RUN_MODE — EDIT HERE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ─── EDIT HERE — the only cell you normally touch ────────────────────────────
REPO_URL           = 'https://github.com/chautuanminh/RESNET_h95.git'
REPO_DIR           = Path('/content/RESNET_h95')                                # local clone (code runs from here)
DATA_ROOT          = Path('/content/drive/MyDrive/usth/Final_Project/dtset/archive')
OUTPUT_ROOT        = Path('/content/drive/Othercomputers/MSI/results/RESNET_h95/runs')
RUN_DIR            = 'doctamper_resnet34_h95_35epochs_comparison'
TAMPER_META        = REPO_DIR / 'tampering_types'                               # committed in the repo
TORCH_HOME         = Path('/content/drive/Othercomputers/MSI/results/RESNET_h95/cache/torch')

# Leakage control — clean pair-hash manifests (Drive mirror of the precomputed archive).
USE_CLEAN_MANIFEST = True
CLEAN_MANIFEST_DIR = Path('/content/drive/Othercomputers/MSI/Approach/ASCFORMER_full/archive_clean_pairhash_manifest_v1')

RUN_MODE           = 'full'    # smoke | train | resume | post_train | full
EPOCHS_OVERRIDE    = None      # int → overrides training.epochs; None keeps the config value (35)
REINSTALL_TORCH    = False     # keep Colab's prebuilt CUDA torch unless you explicitly need a reinstall
TORCH_COMPILE      = 'auto'    # True | False | 'auto' (auto → enabled when effective epochs > 2)
# ──────────────────────────────────────────────────────────────────────

assert RUN_MODE in {'smoke', 'train', 'resume', 'post_train', 'full'}, f'bad RUN_MODE: {RUN_MODE}'
assert DATA_ROOT.exists(), f'data root not found: {DATA_ROOT}'
if USE_CLEAN_MANIFEST:
    assert CLEAN_MANIFEST_DIR.exists(), f'clean manifest dir not found: {CLEAN_MANIFEST_DIR}'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TORCH_HOME.mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(TORCH_HOME)   # cache ImageNet encoder weights on Drive (set before torch import)

print(f'OK  RUN_MODE          : {RUN_MODE}')
print(f'OK  data root         : {DATA_ROOT}')
print(f'OK  output root       : {OUTPUT_ROOT}')
print(f'OK  use_clean_manifest: {USE_CLEAN_MANIFEST}')
print(f'OK  clean manifests   : {CLEAN_MANIFEST_DIR if USE_CLEAN_MANIFEST else "(disabled)"}')
print(f'OK  TORCH_HOME        : {TORCH_HOME}')
print(f'(repo will be cloned to {REPO_DIR} in section 4)')

## 2. GPU detection — warn & fall back, do not hard-assert

Decides `amp_dtype`, `batch_size_candidates`, DataLoader `workers`, and the `torch.compile` mode from the GPU Colab actually allocated. Picked up by section 5 when the config is built.

In [ ]:
import torch

assert torch.cuda.is_available(), 'CUDA is not available — switch to a GPU runtime'
gpu_name       = torch.cuda.get_device_name(0)
total_vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_supported = torch.cuda.is_bf16_supported()
print(f'GPU:  {gpu_name}')
print(f'VRAM: {total_vram_gb:.2f} GB')
print(f'BF16: {bf16_supported}')

if 'H100' in gpu_name:
    AMP_DTYPE        = 'bf16'
    BATCH_CANDIDATES = [64, 80, 96, 112, 128]
    MEMORY_FRACTION  = 0.85
    WORKERS          = 8
    COMPILE_MODE     = 'max-autotune'
    print('✓ H100 detected — BF16, candidates [64..128], workers 8, compile max-autotune')
elif 'A100' in gpu_name:
    AMP_DTYPE        = 'bf16'
    BATCH_CANDIDATES = [56, 60, 64, 72, 80, 96]
    MEMORY_FRACTION  = 0.85
    WORKERS          = 8
    COMPILE_MODE     = 'reduce-overhead'
    print('✓ A100 detected — BF16, candidates [56..96], workers 8, compile reduce-overhead')
elif 'L4' in gpu_name:
    # L4 is Ada Lovelace (compute 8.9): BF16 native, 22.5 GB VRAM.
    # ResNet34-UNet @ 512² uses ~340 MB/sample in the autotune probe.
    AMP_DTYPE        = 'bf16' if bf16_supported else 'fp16'
    BATCH_CANDIDATES = [48, 50, 56, 64]
    MEMORY_FRACTION  = 0.85
    WORKERS          = 4
    COMPILE_MODE     = 'reduce-overhead'
    print(f'✓ L4 detected ({total_vram_gb:.1f} GB) — {AMP_DTYPE.upper()}, candidates [48..64], workers 4')
elif 'V100' in gpu_name:
    AMP_DTYPE        = 'fp16'
    MEMORY_FRACTION  = 0.80
    WORKERS          = 4
    COMPILE_MODE     = 'reduce-overhead'
    if total_vram_gb >= 24:
        BATCH_CANDIDATES = [16, 24, 32, 48, 64]
        print(f'⚠  V100-32GB detected — FP16, candidates [16..64], workers 4')
    else:
        BATCH_CANDIDATES = [8, 16, 24, 32]
        print(f'⚠  V100-16GB detected — FP16, candidates [8..32], workers 4')
elif 'T4' in gpu_name:
    raise RuntimeError(
        f'{gpu_name} only has ~16 GB — insufficient for ResNet34-UNet at 512x512. '
        'Disconnect and request a larger GPU.'
    )
else:
    AMP_DTYPE        = 'bf16' if bf16_supported else 'fp16'
    BATCH_CANDIDATES = [8, 16, 24, 32]
    MEMORY_FRACTION  = 0.80
    WORKERS          = 4
    COMPILE_MODE     = 'reduce-overhead'
    print(f'⚠  Unknown GPU {gpu_name} — defensive defaults: {AMP_DTYPE}, candidates [8..32], workers 4')

## 3. Run mode → stage gating

`RUN_MODE` (set in section 1) selects which stages run:

| RUN_MODE     | smoke | train | resume | eval / failure / tamper | checkpoint used |
|--------------|:-----:|:-----:|:------:|:-----------------------:|-----------------|
| `smoke`      |  ✓   |  –   |  –   |  –  | – |
| `train`      |  ✓   |  ✓ (fresh) |  –   |  –  | – |
| `resume`     |  –   |  ✓   |  ✓   |  –  | `last_checkpoint.pth` |
| `post_train` |  –   |  –   |  –   |  ✓  | `best_model.pth` |
| `full`       |  ✓   |  ✓ (fresh) |  –   |  ✓  | training output |

Autotune overrides below win over the GPU-detected defaults from section 2.

In [ ]:
# ─── Stage gating derived from RUN_MODE (section 1) ─────────────────────────
DO_SMOKE   = RUN_MODE in {'smoke', 'train', 'full'}
DO_TRAIN   = RUN_MODE in {'train', 'resume', 'full'}
DO_RESUME  = RUN_MODE == 'resume'
DO_EVAL    = RUN_MODE in {'post_train', 'full'}
DO_FAILURE = RUN_MODE in {'post_train', 'full'}
DO_TAMPER  = RUN_MODE in {'post_train', 'full'}

# ─── Autotune overrides (None → keep GPU-detected values from § 2) ───────────
# Bump these when § 8's autotune picks too low.
#   22 GB (L4):        [24, 32, 48, 56, 64] @ 0.85 → picks 48
#   40 GB (A100):      [16, 24, 32, 48, 64] @ 0.80 → picks 64
#   40 GB, aggressive: [48, 64, 80, 96]     @ 0.86 → picks 80–96
BATCH_CANDIDATES_OVERRIDE = None   # list[int]   e.g. [32, 48, 56, 64]
MEMORY_FRACTION_OVERRIDE  = None   # float 0..1  e.g. 0.88

print(f'RUN_MODE   = {RUN_MODE}')
print(f'DO_SMOKE   = {DO_SMOKE}')
print(f'DO_TRAIN   = {DO_TRAIN}  (resume={DO_RESUME})')
print(f'DO_EVAL    = {DO_EVAL}')
print(f'DO_FAILURE = {DO_FAILURE}')
print(f'DO_TAMPER  = {DO_TAMPER}')
print(f'EPOCHS_OVERRIDE           = {EPOCHS_OVERRIDE}')
print(f'BATCH_CANDIDATES_OVERRIDE = {BATCH_CANDIDATES_OVERRIDE}')
print(f'MEMORY_FRACTION_OVERRIDE  = {MEMORY_FRACTION_OVERRIDE}')

## 4. Clone repo from GitHub & install dependencies

Clones (or pulls) `chautuanminh/RESNET_h95` into `/content/RESNET_h95`, then installs requirements. By default `torch` / `torchvision` are left out of the install so Colab's prebuilt CUDA wheels stay intact (set `REINSTALL_TORCH = True` in section 1 to override). Finally makes the clone importable.

In [ ]:
import os, sys, subprocess

# Clone once, else pull latest.
if not REPO_DIR.exists():
    print(f'Cloning {REPO_URL} → {REPO_DIR} ...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'Repo present — pulling latest in {REPO_DIR} ...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

assert (REPO_DIR / 'requirements.txt').exists(), f'requirements.txt missing in clone: {REPO_DIR}'
assert TAMPER_META.exists(), f'tamper metadata missing in clone: {TAMPER_META}'

# Install deps. Keep Colab's prebuilt CUDA torch/torchvision unless REINSTALL_TORCH.
if REINSTALL_TORCH:
    subprocess.run(['pip', '-q', 'install', '-r', str(REPO_DIR / 'requirements.txt')], check=True)
else:
    reqs = [ln for ln in (REPO_DIR / 'requirements.txt').read_text().splitlines()
            if ln.strip() and not ln.strip().lower().startswith(('torch', 'torchvision'))]
    Path('/content/reqs_notorch.txt').write_text('\n'.join(reqs) + '\n')
    print('Installing (torch/torchvision left intact):', reqs)
    subprocess.run(['pip', '-q', 'install', '-r', '/content/reqs_notorch.txt'], check=True)

# Make the clone importable.
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f'cwd:         {os.getcwd()}')
print(f'sys.path[0]: {sys.path[0]}')
print('src contents:', sorted(p.name for p in (REPO_DIR / 'src').iterdir()))

## 5. Build the Colab config

Loads the shipped `configs/resnet_h95_config.yaml` from the clone, overrides Drive paths + GPU knobs + leakage manifests + (optionally) `training.epochs`, and writes `/content/resnet_h95_colab.yaml`.

Wired here:
- **Leakage:** `data.use_clean_manifest` + `data.clean_manifest_dir` make `src` draw train/val/eval samples from the clean pair-hash manifests.
- **`torch.compile`:** resolved from `TORCH_COMPILE` (`'auto'` → on when effective epochs > 2).
- **Workers:** `training.num_workers` set from the GPU-detected `WORKERS`.
- **Stage-aware validation:** `tampering_type_analysis.enabled` follows `DO_TAMPER`; `validation.require_outputs` flips to `'auto'` when any post-train stage is skipped.

In [ ]:
from src.config import load_config, deep_get, deep_set, dump_config

config = load_config('configs/resnet_h95_config.yaml')

config = deep_set(config, 'data.root',                str(DATA_ROOT))
config = deep_set(config, 'data.tamper_metadata_dir', str(TAMPER_META))
config = deep_set(config, 'output.root_dir',          str(OUTPUT_ROOT))
config = deep_set(config, 'output.run_dir',           RUN_DIR)

# ─── Leakage control: clean pair-hash manifests ──────────────────────────────
config = deep_set(config, 'data.use_clean_manifest', bool(USE_CLEAN_MANIFEST))
config = deep_set(config, 'data.clean_manifest_dir',  str(CLEAN_MANIFEST_DIR))
print(f'use_clean_manifest = {bool(USE_CLEAN_MANIFEST)}  dir={CLEAN_MANIFEST_DIR if USE_CLEAN_MANIFEST else "(disabled)"}')

# Resolve autotune setup: per-session overrides win over GPU-detected defaults.
batch_candidates_final = list(BATCH_CANDIDATES_OVERRIDE) if BATCH_CANDIDATES_OVERRIDE is not None else BATCH_CANDIDATES
memory_fraction_final  = float(MEMORY_FRACTION_OVERRIDE) if MEMORY_FRACTION_OVERRIDE  is not None else MEMORY_FRACTION
if BATCH_CANDIDATES_OVERRIDE is not None:
    print(f'gpu.batch_size_candidates overridden → {batch_candidates_final}')
if MEMORY_FRACTION_OVERRIDE is not None:
    print(f'gpu.auto_tune_memory_fraction overridden → {memory_fraction_final}')

config = deep_set(config, 'gpu.amp',                       True)
config = deep_set(config, 'gpu.amp_dtype',                 AMP_DTYPE)
config = deep_set(config, 'gpu.tf32',                      True)
config = deep_set(config, 'gpu.channels_last',             True)
config = deep_set(config, 'gpu.cudnn_benchmark',           True)
config = deep_set(config, 'gpu.auto_tune_batch_size',      True)
config = deep_set(config, 'gpu.auto_tune_memory_fraction', memory_fraction_final)
config = deep_set(config, 'gpu.batch_size_candidates',     batch_candidates_final)

config = deep_set(config, 'training.batch_size',                 32)
config = deep_set(config, 'training.num_workers',                int(WORKERS))
config = deep_set(config, 'training.pin_memory',                 True)
config = deep_set(config, 'training.persistent_workers',         True)
config = deep_set(config, 'training.prefetch_factor',            4)
config = deep_set(config, 'training.gradient_accumulation_steps', 1)

if EPOCHS_OVERRIDE is not None:
    config = deep_set(config, 'training.epochs', int(EPOCHS_OVERRIDE))
    print(f'training.epochs overridden → {EPOCHS_OVERRIDE}')

# torch.compile: TORCH_COMPILE in {True, False, 'auto'}; 'auto' → on when epochs > 2.
effective_epochs = int(EPOCHS_OVERRIDE) if EPOCHS_OVERRIDE is not None else int(deep_get(config, 'training.epochs', 35))
if isinstance(TORCH_COMPILE, str) and TORCH_COMPILE.lower() == 'auto':
    compile_enabled = effective_epochs > 2
else:
    compile_enabled = bool(TORCH_COMPILE)
config = deep_set(config, 'gpu.torch_compile',      compile_enabled)
config = deep_set(config, 'gpu.torch_compile_mode', COMPILE_MODE)
print(f'torch.compile = {compile_enabled} (mode={COMPILE_MODE}, effective_epochs={effective_epochs}, flag={TORCH_COMPILE!r})')

# Stage-aware: tell the validator what to expect.
config = deep_set(config, 'tampering_type_analysis.enabled', bool(DO_TAMPER))
all_post_train_on = DO_EVAL and DO_FAILURE and DO_TAMPER
config = deep_set(config, 'validation.require_outputs', True if all_post_train_on else 'auto')

CONFIG_PATH = '/content/resnet_h95_colab.yaml'
Path(CONFIG_PATH).write_text(dump_config(config), encoding='utf-8')
print(f'Wrote {CONFIG_PATH}')
print()
print(dump_config(config))

## 6. Smoke check — LMDB read + H95 stats + mask coverage  *(DO_SMOKE)*

Walks the train folder and each test folder, reads two samples, prints H95 min/mean/max and mask positive ratio. With clean manifests on, the sampled indices come from the manifests — so this also validates that `source_index` rows resolve to real LMDB keys. Runs for `RUN_MODE` in `smoke` / `train` / `full`.

In [ ]:
if DO_SMOKE:
    from src.smoke import run_smoke
    reports = run_smoke(CONFIG_PATH, sample_count=2)
    print(f'\n✓ Smoke OK: {len(reports)} dataset(s) checked')
else:
    print('• smoke skipped for this RUN_MODE')

## 7. GPU utilization monitor *(optional)*

Optional `nvidia-smi` stream (skill Step 7). Set `MONITOR_GPU = True` to start a daemon thread that prints GPU util / memory / temperature every few seconds during training. It runs until the runtime resets.

In [ ]:
MONITOR_GPU = False   # set True to stream GPU util/mem/temp during training

if MONITOR_GPU:
    import subprocess, threading, time

    def _gpu_monitor(interval: float = 5.0):
        while True:
            try:
                out = subprocess.check_output(
                    ['nvidia-smi',
                     '--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu',
                     '--format=csv,noheader,nounits'], text=True).strip()
                util, mem_used, mem_total, temp = [s.strip() for s in out.split(',')]
                print(f'[GPU] util={util}% | mem={mem_used}/{mem_total} MiB | temp={temp}C')
            except Exception:
                pass
            time.sleep(interval)

    threading.Thread(target=_gpu_monitor, daemon=True).start()
    print('GPU monitor started (daemon thread).')
else:
    print('• GPU monitor disabled (set MONITOR_GPU=True to enable)')

## 8. Training  *(DO_TRAIN)*

`run_train` runs `resolve_runtime` → `autotune_batch_size` (real fwd/bwd probe at `[B, 2, 512, 512]`) → model-contract assert → the AMP loop with grad clip + cosine LR, writing `train_metrics.csv`, `val_metrics.csv`, `plots/training_curves.png`, and `checkpoints/{last,best}_*.pth` each epoch.

With clean manifests on, train/val indices come from `clean_internal_train/val_manifest.csv` and a `clean_manifest_sanity_report.json` is written to the run dir. `torch.compile` (when enabled) logs a compile line; the first epoch is slower while it compiles.

- `RUN_MODE='resume'` continues from `<RUN_DIR>/checkpoints/last_checkpoint.pth` on Drive.
- `RUN_MODE='post_train'` skips training and loads `<RUN_DIR>/checkpoints/best_model.pth`.

In [ ]:
CKPT_DIR = OUTPUT_ROOT / RUN_DIR / 'checkpoints'

resume_ckpt = None
if DO_RESUME:
    resume_ckpt = str(CKPT_DIR / 'last_checkpoint.pth')
    assert Path(resume_ckpt).exists(), (
        f"RUN_MODE='resume' but no checkpoint to resume from: {resume_ckpt}. "
        'Run training first, or switch RUN_MODE to "train".'
    )
    print(f'Resuming from: {resume_ckpt}')

if DO_TRAIN:
    from src.train import run_train
    checkpoint = str(run_train(CONFIG_PATH, resume=resume_ckpt))
    print(f'\n✓ Training complete. Best checkpoint: {checkpoint}')
else:
    # post_train (or smoke) — score an already-trained model.
    checkpoint = str(CKPT_DIR / 'best_model.pth')
    if DO_EVAL or DO_FAILURE or DO_TAMPER:
        assert Path(checkpoint).exists(), (
            f'No trained model for post-train stages: {checkpoint}. '
            'Train first (RUN_MODE="full"/"train") before RUN_MODE="post_train".'
        )
        print(f'• Using existing checkpoint: {checkpoint}')
    else:
        print('• No training and no post-train stages for this RUN_MODE.')

## 9. Evaluation  *(DO_EVAL)*

`src.evaluate.run_evaluate` runs the model on TestingSet, FCD, SCD (clean rows only when manifests are on), writing per-image metrics, threshold sweeps, official 0.5 + best-threshold tables, diagnostic example panels, and bar-chart plots.

In [ ]:
if DO_EVAL:
    from src.evaluate import run_evaluate
    eval_rows = run_evaluate(CONFIG_PATH, checkpoint)
    print(f'\n✓ Evaluation complete. {len(eval_rows)} test-set row(s).')
else:
    print('• eval skipped for this RUN_MODE')

## 10. Failure case analysis  *(DO_FAILURE)*

`src.failure_analysis.run_failure_analysis` ranks the worst-K predictions per test set (from the clean per-image metrics), writes summaries by category / reason / severity, and saves the worst-200 visualisations.

In [ ]:
if DO_FAILURE:
    from src.failure_analysis import run_failure_analysis
    run_failure_analysis(CONFIG_PATH, checkpoint)
    print('\n✓ Failure analysis complete.')
else:
    print('• failure analysis skipped for this RUN_MODE')

## 11. Tamper-type analysis  *(DO_TAMPER)*

`src.tampering_type_analysis.run_tampering_type_analysis` groups every prediction by the metadata-assigned tamper type (with heuristic fallback) and writes per-type metrics, threshold sweeps, and best-threshold tables.

In [ ]:
if DO_TAMPER:
    from src.tampering_type_analysis import run_tampering_type_analysis
    run_tampering_type_analysis(CONFIG_PATH, checkpoint)
    print('\n✓ Tamper-type analysis complete.')
else:
    print('• tamper-type analysis skipped for this RUN_MODE')

## 12. Summary + validate

Always-on wrap-up: writes the run-level `RES_SUMMARY.md` / `output.md`, then calls `validate_experiment`. With the stage-aware config from § 5, the validator only enforces the artefacts that the stages you ran actually produced.

In [ ]:
from src.output_summary import write_output_summary
from src.validate_experiment import validate_experiment

write_output_summary(CONFIG_PATH)
ok, errors = validate_experiment(CONFIG_PATH)
if ok:
    print('✓ validate_experiment: PASS')
else:
    print('⚠ validate_experiment: FAIL')
    for err in errors:
        print(f'  - {err}')
print(f'\nAll outputs under: {OUTPUT_ROOT / RUN_DIR}')

## 13. Notes

### Leakage control
Sample selection is driven by the clean pair-hash manifests in `CLEAN_MANIFEST_DIR`
(`use_clean_manifest=True`): train/val come from `clean_internal_train/val_manifest.csv`,
eval from `clean_eval_{TestingSet,FCD,SCD}_manifest.csv`. Counts: train 99618 / val 10000 /
TestingSet 21122 / FCD 2000 / SCD 18000. A `clean_manifest_sanity_report.json` is written to
the run dir and the run fails fast if any `pair_sha256` overlap is detected. Set
`USE_CLEAN_MANIFEST=False` only to reproduce the old (leaky) index-based split.

### Where outputs land
Under `/content/drive/Othercomputers/MSI/results/RESNET_h95/runs/<RUN_DIR>/`:
`training.log`, `config_resolved.yaml`, `clean_manifest_sanity_report.json`,
`batch_size_autotune.csv`, `train_metrics.csv`, `val_metrics.csv`, `plots/`,
`checkpoints/{best_model,last_checkpoint}.pth`, per-image metrics / threshold sweeps /
examples, and `failure_case_analysis/` + `tampering_type_analysis/` when those stages run.

### Resuming / post-train
Checkpoints persist on Drive — set `RUN_MODE='resume'` to continue from
`last_checkpoint.pth`, or `RUN_MODE='post_train'` to score `best_model.pth` with no training.

### GPU optimization
`TORCH_HOME` caches the encoder weights on Drive. `torch.compile` follows `TORCH_COMPILE`
(`'auto'` → on when epochs > 2; first epoch compiles, then accelerates). `WORKERS`, AMP dtype,
TF32, channels-last, and cuDNN benchmark are GPU-detected. Batch size is autotuned for
ResNet34-UNet @ 512² — do not paste the 128–256 sizes meant for 224² classification.

### Code source
Code is canonical on GitHub (`https://github.com/chautuanminh/RESNET_h95`); the clone cell
pulls latest each run. The local `archive_clean_pairhash_manifest_v1/` mirror is gitignored.